# **Post-Read 3: Advanced ANOVA, Chi-Square, and Correlation with the Credit Card Customer Dataset**

*Welcome to **Post-Read 3**, a supplementary resource for **Main Lecture 3** on ANOVA, Chi-Square Tests, and Correlation. We’ll explore advanced or optional topics—like Two-Way ANOVA, non-parametric tests, and deeper Chi-Square insights—using a **Credit Card Customer dataset** rather than the previous Medical Cost dataset.*

In [ ]:
!wget https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/103/691/original/Credit_Card_Customer_Data.csv

--2025-01-21 05:07:01--  https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/103/691/original/Credit_Card_Customer_Data.csv
Resolving d2beiqkhq929f0.cloudfront.net (d2beiqkhq929f0.cloudfront.net)... 3.167.84.28, 3.167.84.9, 3.167.84.148, ...
Connecting to d2beiqkhq929f0.cloudfront.net (d2beiqkhq929f0.cloudfront.net)|3.167.84.28|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16483 (16K) [text/plain]
Saving to: ‘Credit_Card_Customer_Data.csv’

Credit_Card_Custome 100%[===================>]  16.10K  --.-KB/s    in 0s      

2025-01-21 05:07:01 (212 MB/s) - ‘Credit_Card_Customer_Data.csv’ saved [16483/16483]



In [ ]:
import pandas as pd

df = pd.read_csv('Credit_Card_Customer_Data.csv')
df.head()

,Sl_No,Customer Key,Avg_Credit_Limit,Total_Credit_Cards,Total_visits_bank,Total_visits_online,Total_calls_made
0,1,87073,100000,2,1,1,0
1,2,38414,50000,3,0,10,9
2,3,17341,50000,7,1,3,4
3,4,40496,30000,5,1,1,4
4,5,47437,100000,6,0,12,3


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 660 entries, 0 to 659
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Sl_No                660 non-null    int64
 1   Customer Key         660 non-null    int64
 2   Avg_Credit_Limit     660 non-null    int64
 3   Total_Credit_Cards   660 non-null    int64
 4   Total_visits_bank    660 non-null    int64
 5   Total_visits_online  660 non-null    int64
 6   Total_calls_made     660 non-null    int64
dtypes: int64(7)
memory usage: 36.2 KB


---

## 1. Introduction

### 1.1 Dataset & Business Context

We’re working with a **Credit Card Customer** dataset containing **660** entries. It has **7 columns**:

1. <font color="skyblue">Sl_No</font> (int64)  
2. <font color="skyblue">Customer Key</font> (int64)  
3. <font color="skyblue">Avg_Credit_Limit</font> (int64)  
4. <font color="skyblue">Total_Credit_Cards</font> (int64)  
5. <font color="skyblue">Total_visits_bank</font> (int64)  
6. <font color="skyblue">Total_visits_online</font> (int64)  
7. <font color="skyblue">Total_calls_made</font> (int64)

**Business Setting**: A credit card company wants to understand customer usage patterns, credit limits, and overall engagement. This helps them tailor services, improve customer satisfaction, and detect potential areas for cost optimization or targeted marketing.

### 1.2 Goals of This Notebook

- **Reinforce & expand** Main Lecture 3 topics (ANOVA, Chi-Square, Correlation).
- Introduce **advanced/optional** concepts:
  - <font color="magenta">Two-Way ANOVA</font>
  - <font color="magenta">Non-Parametric Tests</font> (Kruskal-Wallis, Mann-Whitney)
  - <font color="magenta">Advanced Chi-Square</font> considerations
  - <font color="magenta">Correlation</font> expansions (e.g., Spearman, partial correlations)
- Provide **short code demonstrations** using the **Credit Card Customer** dataset.
- Offer **mini-exercises** to test your understanding.

---

## 3. Two-Way ANOVA

### 3.1 Conceptual Background

A <font color="magenta">Two-Way ANOVA</font> tests the effect of **two different categorical factors** on a continuous dependent variable—and checks for interaction effects between those factors.

For instance, if we wanted to see how **Avg_Credit_Limit** varies by:
1. **Total_Credit_Cards** (factor 1)  
2. Some **categorized** version of `Total_visits_bank` (factor 2)

We could run a two-way ANOVA to see:
- Does **Avg_Credit_Limit** significantly differ with **Total_Credit_Cards** (levels)?
- Does **Avg_Credit_Limit** differ with categories of **Total_visits_bank**?
- Is there an **interaction** between those two factors?

### 3.2 Example Code: Creating Two Factors and Running Two-Way ANOVA

First, let’s create **categorical factors** from our data (for demonstration):

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Factor 1: Total_Credit_Cards (group into 3 categories: low, medium, high)
# This is artificial grouping for demonstration.
df['Credit_Card_Group'] = pd.cut(df['Total_Credit_Cards'],
                                 bins=[-1, 2, 4, df['Total_Credit_Cards'].max()],
                                 labels=['LowCards', 'MedCards', 'HighCards'])

# Factor 2: visits_bank (group into 2 categories: low visits vs. high visits)
median_visits = df['Total_visits_bank'].median()
df['Bank_Visits_Group'] = np.where(df['Total_visits_bank'] > median_visits, 'HighVisits', 'LowVisits')

df[['Total_Credit_Cards', 'Credit_Card_Group', 'Total_visits_bank', 'Bank_Visits_Group']].head()

,Total_Credit_Cards,Credit_Card_Group,Total_visits_bank,Bank_Visits_Group
0,2,LowCards,1,LowVisits
1,3,MedCards,0,LowVisits
2,7,HighCards,1,LowVisits
3,5,HighCards,1,LowVisits
4,6,HighCards,0,LowVisits


Now, we can use libraries like `statsmodels` to run a two-way ANOVA:

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Build a formula: "Avg_Credit_Limit ~ C(Credit_Card_Group) + C(Bank_Visits_Group) + C(Credit_Card_Group):C(Bank_Visits_Group)"
model = ols('Avg_Credit_Limit ~ C(Credit_Card_Group) * C(Bank_Visits_Group)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)  # Type II ANOVA table
anova_table

,sum_sq,df,F,PR(>F)
C(Credit_Card_Group),2.098285e+11,2.0,115.048580,1.511787e-43
C(Bank_Visits_Group),9.291027e+09,1.0,10.188507,1.480785e-03
C(Credit_Card_Group):C(Bank_Visits_Group),1.754105e+11,2.0,96.177275,2.386073e-37
Residual,5.973027e+11,655.0,NaN,NaN


**Interpretation**:
- **p-values** for each main effect (Credit_Card_Group, Bank_Visits_Group).
- **p-value** for the **interaction** (Credit_Card_Group × Bank_Visits_Group).
- If an interaction is significant, the effect of one factor depends on the level of the other factor.

---

## 4. Non-Parametric Alternatives

### 4.1 When to Use Non-Parametric Tests?

- If **ANOVA assumptions** (normality, homogeneity of variance) are **not met**.
- If data is ordinal or heavily skewed, or sample sizes are very small.

### 4.2 Kruskal-Wallis for >2 Groups

<font color="magenta">Kruskal-Wallis</font> is the non-parametric analogue to one-way ANOVA for **comparing more than 2 groups**. For instance, if we check `Avg_Credit_Limit` across the three `Credit_Card_Group` categories without assuming normality:

In [ ]:
from scipy.stats import kruskal

low_group = df[df['Credit_Card_Group'] == 'LowCards']['Avg_Credit_Limit']
med_group = df[df['Credit_Card_Group'] == 'MedCards']['Avg_Credit_Limit']
high_group = df[df['Credit_Card_Group'] == 'HighCards']['Avg_Credit_Limit']

stat, p_value = kruskal(low_group, med_group, high_group)
print(f"Kruskal-Wallis H-stat: {stat:.3f}, p-value: {p_value:.6f}")

Kruskal-Wallis H-stat: 141.762, p-value: 0.000000


### 4.3 Mann-Whitney U for 2 Groups

If you only have **2 groups** (e.g., `HighVisits` vs `LowVisits` on `Avg_Credit_Limit`), you can use <font color="magenta">Mann-Whitney U</font> test:


In [ ]:
high_visits_limit = df[df['Bank_Visits_Group'] == 'HighVisits']['Avg_Credit_Limit']
low_visits_limit = df[df['Bank_Visits_Group'] == 'LowVisits']['Avg_Credit_Limit']

u_stat, p_val = stats.mannwhitneyu(high_visits_limit, low_visits_limit, alternative='two-sided')
print(f"Mann-Whitney U: {u_stat}, p-value: {p_val:.6f}")

Mann-Whitney U: 63151.0, p-value: 0.000092


---

## 5. Parametric vs. Non-Parametric

In summary:
- **Parametric tests**: Assume normality and often equal variances. They can be more **powerful** if assumptions hold.
- **Non-parametric tests**: More **robust** to outliers, skewed data, or ordinal data but may have **less power** if the parametric assumptions were valid.

💡 **Tip**: Always **check assumptions** (e.g., via histograms, Q-Q plots, Shapiro-Wilk test, Levene’s test for homogeneity) before deciding.

---

## 6. Advanced Chi-Square Insights

### 6.1 Chi-Square Test for Independence

In **Main Lecture 3**, you learned the **Chi-Square test** for independence in a contingency table (for two categorical variables). Here are a few **nuances**:

- **Expected cell counts** should generally be $\ge 5$. If many are < 5, consider combining categories or using **Fisher’s Exact Test** (especially in 2×2 tables).
- A **continuity correction** can be applied in 2×2 cases, but is less common with larger tables.

### 6.2 Example: Chi-Square on Binned Variables

Let’s say we want to see if `Total_Credit_Cards` (binned) is independent of a **new** categorical variable dividing `Total_visits_online` into “HighOnline” vs. “LowOnline”:

In [ ]:
# Create the second categorical: "Online_Visits_Group"
median_online = df['Total_visits_online'].median()
df['Online_Visits_Group'] = np.where(df['Total_visits_online'] > median_online, 'HighOnline', 'LowOnline')

# Contingency table
contingency_table = pd.crosstab(df['Credit_Card_Group'], df['Online_Visits_Group'])

from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(contingency_table)
print("Chi-square value:", chi2)
print("P-value:", p)
print("Degrees of freedom:", dof)
print("Expected frequencies:", expected)

Chi-square value: 138.88664609630106
P-value: 6.936621460054009e-31
Degrees of freedom: 2
Expected frequencies: [[ 40.62727273  82.37272727]
 [ 67.38181818 136.61818182]
 [109.99090909 223.00909091]]


- If **p < 0.05**, we might conclude there is an **association** between these two categorical variables (i.e., they are not independent).

---

## 7. Correlation Refinements

### 7.1 Pearson vs. Spearman

- **Pearson correlation** (parametric) measures **linear** relationship; sensitive to outliers.
- **Spearman rank correlation** (non-parametric) measures **monotonic** relationship; more robust to outliers or non-linear patterns.

### 7.2 Code Example

In [ ]:
# Compare correlation methods for Avg_Credit_Limit vs. Total_calls_made
pearson_corr, pearson_pval = stats.pearsonr(df['Avg_Credit_Limit'], df['Total_calls_made'])
spearman_corr, spearman_pval = stats.spearmanr(df['Avg_Credit_Limit'], df['Total_calls_made'])

print(f"Pearson correlation: {pearson_corr:.3f}, p-value: {pearson_pval:.6f}")
print(f"Spearman correlation: {spearman_corr:.3f}, p-value: {spearman_pval:.6f}")

Pearson correlation: -0.414, p-value: 0.000000
Spearman correlation: -0.451, p-value: 0.000000


- **Compare** the two correlations. If they differ substantially, it might suggest a non-linear relationship or outliers.


### 7.3 Partial Correlation (Optional)

If you want to **control for a third variable** (say `Total_visits_bank`), you could explore **partial correlation**. Libraries like `pingouin` (`pip install pingouin`) provide a straightforward approach, but we won’t delve into detailed code here—just note it as an **advanced** concept.

---

## 8. Mini-Case Study Example

Let’s outline a **brief scenario** using these concepts:

1. **Goal**: Investigate whether different levels of `Total_Credit_Cards` and `Bank_Visits_Group` affect `Avg_Credit_Limit` using a <font color="magenta">Two-Way ANOVA</font>.
2. **Check** if data meets normality assumptions. If not, try <font color="magenta">Kruskal-Wallis</font> on one factor or use a repeated approach for each group.
3. **Explore** if there’s an association between newly-created categories (`Credit_Card_Group` and `Online_Visits_Group`) via **Chi-Square**.
4. **Check** correlation between continuous metrics (e.g., `Avg_Credit_Limit` and `Total_calls_made`) with both **Pearson** and **Spearman**.

By doing these steps, we gather a more **comprehensive** view of the data and understand relationships among variables—potentially guiding business decisions like marketing strategies for different segments.

---

## 9. Exercises & Reflection

Try these **mini-quizzes** to cement your understanding:

### 9.1 Two-Way ANOVA Interpretation
**Q:** If your two-way ANOVA shows a **significant interaction** between `Credit_Card_Group` and `Bank_Visits_Group`, what does that imply about `Avg_Credit_Limit`?

<details>
<summary><em>Hint/Answer</em></summary>
It indicates that the effect of one factor (e.g., Credit_Card_Group) on the average credit limit *depends on* the level of the other factor (Bank_Visits_Group), suggesting a *non-additive* relationship.
</details>

---

### 9.2 Non-Parametric Choice
**Q:** When would you prefer **Kruskal-Wallis** over a standard one-way ANOVA?

<details>
<summary><em>Hint/Answer</em></summary>
When the normality or equal variance assumptions are violated, or the data is ordinal/skewed and ANOVA might not be appropriate.
</details>

---

### 9.3 Chi-Square Assumptions
**Q:** Why is it important to check **expected frequencies** in each cell before interpreting a Chi-Square test?

<details>
<summary><em>Hint/Answer</em></summary>
If many cells have low expected counts (<5), the Chi-Square approximation may be unreliable. You might need to combine categories or use alternative methods (Fisher’s Exact for a 2×2 table).
</details>

---

### 9.4 Pearson vs. Spearman
**Q:** If Pearson correlation between `Avg_Credit_Limit` and `Total_calls_made` is very low, but Spearman correlation is quite high, what might that tell you?

<details>
<summary><em>Hint/Answer</em></summary>
It may suggest a *non-linear* but *monotonic* relationship or the presence of outliers affecting the Pearson result. Spearman's rank correlation captures monotonic trends better.
</details>

---